# **泛化Tiling设计**
如果开发者想支持“一类” 算子，要能适合任何合法的数据类型、数据形状，甚至适合多种昇腾AI处理器型号，这种场景，称之为算子的泛化。在Ascend C算子工程化开发时，Kernel侧代码实现决定了算子的运算逻辑，Host侧的Tiling实现决定了数据的分块策略，所以要想开发一个泛化的Ascend C算子，就需要完成泛化Tiling的开发。
我们将按照以下结构，带你学习泛化Tiling开发：
- 环境准备
- Tiling概念
- Tiling设计思路介绍
- 泛化Tiling设计示例
- 课后实践 
---

## **1. 环境准备**
本文所有内容均存放于 `Sources/02.16` 文件夹。
在开始编写直调示例前，先初始化 Jupyter 环境，并创建代码目录，保证后续代码可以正常编译运行。

In [ ]:
!mkdir -p Sources/02.16

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("Environment initialization process completed successfully!")

--- 
## **2. Tiling概念**
在NPU架构中，AI Core的片上存储容量有限，如果输入数据量较大时，会存在无法一次性加载算子全部输入、输出与中间计算数据的情况。因此需要将完整数据按计算粒度切分为多个小块，通过 “搬运→计算→回写” 的循环流水线完成全量计算，这个数据分块、分段计算的过程，就是 Tiling。<br>
简单来说：受限于片上存储大小，把 “一次装不下、算不完” 的大数据，拆成多块小数据分批处理，就是 Tiling。<br>

接下来，我们通过一个直观的示例来理解tiling机制的核心作用：  
- 设定场景：假设某 AI Core 的片上存储容量上限为 10 个数据元素，我们需要完成一个长度为 10 的向量加法运算（输入张量 A + 输入张量 B = 输出张量 C）。  
- 核心约束：运算过程中，输入张量（A、B）与输出张量（C）需同时驻留在 AI Core 的片上存储中，因此每次计算时，三者的总元素个数必须≤片上存储容量（10 个元素）。  
- 分片计算逻辑：基于上述约束，单次可处理的最大数据块大小为 `max_once_num = 10 // 3 = 3`（3 个元素对应A、B、C各3个，总占用9个存储位置，符合容量限制）；整个向量加法需切分为 `tile_num = (10 + 3 - 1) // 3 = 4` 次完成（向上取整），最终运算结果与整段数据直接计算完全一致。 

整体执行流程如图：  
<img src="./images/tile_animation_slow.gif" alt="tiling" width="700px" />

常见的Tiling术语有如下：

➤ 每次搬运的那一部分数据块，叫做**Tiling块**  
➤ 根据算子中不同输入形状确定搬入基本块大小的相关算法，叫做**Tiling算法**（或Tiling策略）  
➤ 算子中实现Tiling算法的函数（一般定义在host侧的tiling头文件中），叫做**Tiling函数**（或Tiling Function）  

---
## **3. Tiling设计思路**
受硬件架构约束，在对输入数据进行切分与调度时，通常需要遵循以下设计原则，以保障算子性能与执行正确性：

 ### **3.1 内存对齐原则**  
AI Core 的Unified Buffer存在物理层面的存储约束，要求其上的数据空间必须保持32字节对齐。
- 若输入数据长度不满足32字节对齐，需将其长度向**上对齐至32字节**，并以此作为总长度参与后续计算。
- 所有Tiling相关的计算逻辑，均应以**32**字节为最小粒度单位进行。 

 ### **3.2 访存优化原则**  
AI Core 与外部存储（Global Memory）间的数据交互会引入显著的性能开销，频繁的数据搬运极易成为性能瓶颈。
- 核心目标：充分利用 Unified Buffer 空间，通过增大单次搬运的数据块大小，最大限度减少数据搬运的总次数。 
 
 ### **3.3 多核均衡原则**  
昇腾 AI 处理器集成了多个 AI Core，为充分释放硬件算力，需进行合理的任务调度。
- 核心目标：均衡利用多核计算能力，将整体计算任务均匀地分配至各个 AI Core 上，避免出现算力闲置或负载不均的情况。

<img src="./images/tiling_partition_schematic.png"  alt="tiling_partition_schematic" />

数据切分示意如上图所示，将长度为TOTAL_LENGTH的算子输入分配到多个核上进行计算，每个核上计算的数据长度为BLOCK_LENGTH。对于每个核的计算数据，基于Local Memory的大小进一步切分，切分数据块的个数为TILE_NUM，得到的每个数据块的长度为TILE_LENGTH。

根据每个核计算的数据量是否相同、核内每个数据块的数据量是否相同，切分策略可能会存在以下几种场景：
- 核间均分，核内均分：每个核处理的数据量相同，核内每个数据块的数据量相同。在此场景中，通过多核Tiling将数据均匀分配到各个核上执行，每个核上每次计算的数据长度相同。  

- 核间均分，核内不均分：每个核处理的数据量相同，核内各数据块的数据量不完全相同。此场景基于多核Tiling，核内数据不能切分为多个数据量相同且32字节对齐的数据块，需要通过尾块Tiling处理尾块数据的计算。  

- 核间不均分，核内均分：每个核处理的数据量不同，核内每个数据块的数据量相同。在此场景中，通过尾核Tiling的处理解决数据无法在各核间均匀分配的问题。  

- 核间不均分，核内不均分：每个核处理的数据量不同，核内各数据块的数据量不完全相同。该场景下需要同时考虑尾核&尾块，处理多核间及核内数据的合理切分。  

### **3.4 Tiling设计示例**

以状为(1,660)的half类型算子输入数据，要分配到4个aicore上完成Add计算为例，我们来逐步完成Tiling设计。

#### **3.4.1 32字节对齐**
Add算子输入为`shape=(1,660)`、`dtype=half`的张量（`half`单元素占2B）。因硬件要求数据对齐到32B，`660×2=1320B`无法被32整除，需将输入从660向上补齐至32B对齐，计算逻辑如下：
```text
(660 × 2 + 32 - 1) // 32 × 32 = 672 × 2 = 1344B
```

<img src="./images/align_32.png" alt="align_32" width="700">


#### **3.4.2 核间数据拆分**
将补齐后的 **42 个 32B 数据块** 分配到 4 个 AI Core 上并行计算：

- 基础分配：`42 // 4 = 10` 个块/核
- 剩余块：`42 % 4 = 2` 个，由前 2 个核各多处理 1 块

最终分配：
- 前 2 个核（大核）：11 块 → `11×32B = 352B`
- 后 2 个核（小核）：10 块 → `10×32B = 320B`

<img src="./images/core_data_split.png" alt="tiling_32" width="700">

#### **3.4.3 核内数据切分**
假设UB（Unified Buffer）容量限制为768B，单核单次处理数据量受此约束，需按UB大小对核内数据进行批次拆分。  

Add算子包含2个输入（x/y）和1个输出（z），且三者数据大小一致，因此UB空间均分后：  
- 单路数据最大可用UB空间：`768 // 3 = 256B`  
- 对应32B数据块数量：`256 ÷ 32 = 8` 块（即单核单次最多处理8个32B数据块）  

基于单批次上限，各核数据拆分如下：  
- 前2个核（共11个32B块）：第1批处理8块，第2批处理3块  
- 后2个核（共10个32B块）：第1批处理8块，第2批处理2块  

<img src="./images/kernel_data_split.png" alt="tiling_32" width="700">

#### **3.4.4 Tiling 结构体定义**
为存储 Tiling 相关参数，我们基于前述的核间拆分、UB 批次拆分逻辑，我们需要以下字段：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 800px; border: 1px solid #ddd;">
  <!-- 表头行 -->
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5;">结构体字段</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5;">对应数值</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5;">计算逻辑/业务含义</td>
  </tr>
  <!-- 行1 -->
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><code>smallCoreDataNum</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">320B</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">小核（后2个核）总数据量：10个32B块 × 32B/块 = 320B</td>
  </tr>
  <!-- 行2 -->
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><code>bigCoreDataNum</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">352B</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">大核（前2个核）总数据量：11个32B块 × 32B/块 = 352B</td>
  </tr>
  <!-- 行3 -->
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><code>finalBigTileNum</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">2</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">大核批次次数：单批次最多8块（256B），11块需分2批（8块+3块）</td>
  </tr>
  <!-- 行4 -->
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><code>finalSmallTileNum</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">2</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">小核批次次数：单批次最多8块（256B），10块需分2批（8块+2块）</td>
  </tr>
  <!-- 行5 -->
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><code>tileDataNum</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">256B</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">单核单次最大搬运量：UB 768B ÷ 3（2输入+1输出）= 256B（8个32B块）</td>
  </tr>
  <!-- 行6 -->
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><code>smallTailDataNum</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">64B</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">小核最后一批数据量：2个32B块 × 32B/块 = 64B</td>
  </tr>
  <!-- 行7 -->
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><code>bigTailDataNum</code></td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">96B</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">大核最后一批数据量：3个32B块 × 32B/块 = 96B</td>
  </tr>
  <!-- 行8 -->
  <tr>
    <td style="padding: 8px;"><code>tailBlockNum</code></td>
    <td style="padding: 8px;">2</td>
    <td style="padding: 8px;">大核个数：42个32B块 ÷ 4核，余数为2，前2个核为大核</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

<br>

基于上述字段定义 Tiling 结构体：
```cpp
struct TilingDataTemplate {
    uint32_t smallCoreDataNum;  // 小核处理的总数据量（元素个数）
    uint32_t bigCoreDataNum;    // 大核处理的总数据量（元素个数）
    uint32_t finalBigTileNum;   // 大核数据搬运总批次次数
    uint32_t finalSmallTileNum; // 小核数据搬运总批次次数
    uint32_t tileDataNum;       // 单核单次可搬运数据量（元素个数）
    uint32_t smallTailDataNum;  // 小核最后一批处理数据量（元素个数）
    uint32_t bigTailDataNum;    // 大核最后一批处理数据量（元素个数）
    uint32_t tailBlockNum;      // 大核个数（余数分配的核数量）
};

---
## **4. 泛化算子实现**
接下来我们将基于上述 Tiling 设计思路，完成 AddCustomTemplate 的泛化开发。

本次算子代码将按照如下目录结构存放：

```
├── 2.16
│   ├── scripts
│   │   ├── gen_data.py                // 输入数据和真值数据生成脚本
│   │   └── verify_result.py           // 验证输出数据和真值数据是否一致的验证脚本
│   ├── CMakeLists.txt                 // 编译工程文件
│   ├── data_utils.h                   // 数据读入写出函数
│   ├── add_custom_template.asc        // host侧tiling计算 + kernel侧实现
│   └── run.sh                         // 运行脚本
```
我们本次学习的重点是Tiling的实现，因此只需要关注 `add_custom_template.asc` 文件。

In [ ]:
!cp -rf ./src/02.16 ./Sources

由于泛化Tiling实现时需要获取AiCore的UB大小，以及可能需要获取的数据类型的字节数，所以需要引用platform_ascendc.h。实现Kernel侧算子类时，需引用相关固定头文件kernel_operator.h；同时因Tiling设计阶段未考虑开启Double Buffer模式，此处需将BUFFER_NUM定义为1。

In [ ]:
%%writefile Sources/02.16/add_custom_template.asc
#include <algorithm>
#include <cstdint>
#include <iostream>
#include <vector>

#include "data_utils.h"
#include "acl/acl.h"
#include "kernel_operator.h"
#include "tiling/platform/platform_ascendc.h"

constexpr int32_t BUFFER_NUM = 1;

struct TilingDataTemplate {
    uint32_t smallCoreDataNum;  // 小核处理的总数据量（元素个数）
    uint32_t bigCoreDataNum;    // 大核处理的总数据量（元素个数）
    uint32_t finalBigTileNum;   // 大核数据搬运总批次次数
    uint32_t finalSmallTileNum; // 小核数据搬运总批次次数
    uint32_t tileDataNum;       // 单核单次可搬运数据量（元素个数）
    uint32_t smallTailDataNum;  // 小核最后一批处理数据量（元素个数）
    uint32_t bigTailDataNum;    // 大核最后一批处理数据量（元素个数）
    uint32_t tailBlockNum;      // 大核个数（余数分配的核数量）
};


### **4.1 Host 侧 Tiling 计算**
首先需要完成Host侧的Tiling实现，该部分逻辑决定了每个核处理的数据量，以及核内数据的切分方式。我们通过函数`CalcTiling` 完成 Tiling 参数计算。

#### **4.1.1 获取输入大小及核数**
根据Tiling设计多核均衡原则，每个核处理的数据量应尽可能相等。所以我们需要先获取输入数据大小以及可以使用的核数。
- GetCoreNum：获取当前环境的核数

Host侧根据输入元素个数 `inputNum`、单元素字节数 `typeLength`，并计算输入总字节数：
```cpp
inputLength = inputNum * typeLength;
```

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc

template <typename T>
void CalcTiling(uint32_t inputNum, TilingDataTemplate &tiling, uint32_t &numBlocks)
{
    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    uint32_t coreNum = ascendcPlatform->GetCoreNum();

    const uint32_t typeLength = static_cast<uint32_t>(sizeof(T));
    const uint32_t inputLength = inputNum * typeLength;


#### **4.1.2 计算每个核处理的数据量**
结合Tiling设计的内存对齐原则，输入数据需向上对齐至32B的整数倍，且所有数据块均以32B为最小处理单位。基于此，核数分配与核内数据块的切分规则设计如下：
1. 先将输入数据的总大小向上对齐至32B整数倍，得到对齐后的总32B数据块数；  

2. 以每个核至少分配1个32B数据块为基准，判断是否启用全部核数：若对齐后的数据块总量过小，核数不足时则至少启用1个核完成计算；  

3. 确定实际使用的核数后，计算核均基础处理的32B数据块数，对无法均分的剩余数据块，采用前N核补块的方式分配：让前tailBlockNum个核各多处理1个32B数据块，其余核按基础数处理；  

4. 按补块规则划分核类型：多处理1个32B 数据块的核称为大核，按基础数处理的核称为小核。  

**参数说明如下：**

- everyCoreInputBlockNum： 每个核处理的32B数据块个数

- tailBlockNum： 前tailBlockNum个核每个核多计算一个32B数据块

<img src="./images/input_alignment.png"  alt="input_alignment" width="800" height="250"/>

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc
    const uint32_t BLOCK_SIZE = 32;
    const uint32_t inputLengthAlign32 = ((inputLength + BLOCK_SIZE - 1) / BLOCK_SIZE) * BLOCK_SIZE;
    coreNum = std::min(coreNum, inputLengthAlign32 / BLOCK_SIZE);
    coreNum = std::max(coreNum, 1U);
    numBlocks = coreNum;

    const uint32_t everyCoreInputBlockNum = inputLengthAlign32 / BLOCK_SIZE / coreNum;
    const uint32_t tailBlockNum = (inputLengthAlign32 / BLOCK_SIZE) % coreNum;


#### **4.1.3 核内数据切分**
结合访存优化原则，设计核心为尽可能占满单个核的 UB 内存，以此最大化访存效率。具体计算逻辑如下：  

1. 先获取单个核的可用UB内存总大小；  
2. 针对AddCustomTemplate算子，其包含两个输入、一个输出，因此核上会同时存在输入x、输入y、输出z共3个Buffer块；  
3. 基于核可用UB总大小与3个Buffer块的数量，即可计算出单个Buffer最多可容纳的32B数据块个数，以及对应的元素个数。

**参数说明如下：**
- GetCoreMemSize： 获取硬件平台存储空间的内存大小

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc
    uint64_t ubSize = 0;
    ascendcPlatform->GetCoreMemSize(platform_ascendc::CoreMemType::UB, ubSize);
    const uint32_t ubDataNumber = 3;
    const uint32_t tileBlockNum = (ubSize / BLOCK_SIZE ) / ubDataNumber;
    const uint32_t tileDataNum = tileBlockNum * BLOCK_SIZE / typeLength;


针对小核的运算调度，需结合其分配到的总数据量、核内单次可运算的数据量，计算核内的循环运算次数；若总数据量无法被单次运算量整除，需额外执行一次循环处理剩余数据。

**参数说明如下：**
- smallTileNum： 每个核分到数据不能一次运算完成时需要循环计算的次数

- smallTailDataNum： 最后一次循环需要处理的数据量

<img src="./images/intra_kernel_partition.png"  alt="intra_kernel_partition" />

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc
    const uint32_t smallCoreDataNum = everyCoreInputBlockNum * BLOCK_SIZE / typeLength;
    const uint32_t smallTileNum = everyCoreInputBlockNum / tileBlockNum;
    const uint32_t finalSmallTileNum = (everyCoreInputBlockNum % tileBlockNum) == 0 ? smallTileNum : smallTileNum + 1;
    uint32_t smallTailDataNum = smallCoreDataNum - tileDataNum * smallTileNum;
    smallTailDataNum = smallTailDataNum == 0 ? tileDataNum : smallTailDataNum;


对于大核，因需多处理1个32B数据块，其实际处理的32B数据块总数为everyCoreInputBlockNum + 1

**参数说明如下：**
- bigTileNum：每个核分到数据不能一次运算完成时需要循环计算的次数
- bigTailDataNum：大核最后一次循环需要处理的数据量

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc

    const uint32_t bigCoreInputBlockNum = everyCoreInputBlockNum + 1;
    const uint32_t bigCoreDataNum = bigCoreInputBlockNum * BLOCK_SIZE / typeLength;
    const uint32_t bigTileNum = bigCoreInputBlockNum / tileBlockNum;
    const uint32_t finalBigTileNum = (bigCoreInputBlockNum % tileBlockNum) == 0 ? bigTileNum : bigTileNum + 1;
    uint32_t bigTailDataNum = bigCoreDataNum - tileDataNum * bigTileNum;
    bigTailDataNum = bigTailDataNum == 0 ? tileDataNum : bigTailDataNum;


我们将上述所有计算得到的参数值，逐一赋值至TilingDataTemplate结构体进行存储。至此，便完成了整套Tiling方案的设计与实现，该方案可全面覆盖4种不同的Tiling场景，具备良好的泛化性。

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc

    tiling.smallCoreDataNum = smallCoreDataNum;
    tiling.bigCoreDataNum = bigCoreDataNum;
    tiling.finalBigTileNum = finalBigTileNum;
    tiling.finalSmallTileNum = finalSmallTileNum;
    tiling.tileDataNum = tileDataNum;
    tiling.smallTailDataNum = smallTailDataNum;
    tiling.bigTailDataNum = bigTailDataNum;
    tiling.tailBlockNum = tailBlockNum;
}


### **4.2 Kernel实现**
完成Host侧的开发后，接下来开展 Kernel 侧算子类的实现工作。
#### **4.2.1 定义算子类**
定义算子类，通过模板接收输入和输出的数据类型。

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc

template <typename T>
class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}


声明Init函数完成算子计算参数的初始化，函数入参包含输入输出张量、Tiling切分数据。
1. 先基于大核处理元素数，计算当前核的GlobalMemory地址偏移。

2. 通过当前核索引与tailBlockNum的对比结果判断核类型
    - 若为大核，直接配置算子类的大核规格参数（元素数、切分次数、尾块元素数）；
    
    - 若为小核，则配置小核规格参数，同时重新计算GlobalMemory地址偏移（因初始偏移量基于大核计算）。

3. 最后根据已配置的核规格参数，完成GlobalTensor的初始化与LocalMemory的内存分配。

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t smallCoreDataNum,
                                uint32_t bigCoreDataNum, uint32_t finalBigTileNum,
                                uint32_t finalSmallTileNum, uint32_t tileDataNum,
                                uint32_t smallTailDataNum, uint32_t bigTailDataNum,
                                uint32_t tailBlockNum)
    {
        uint32_t coreNum = AscendC::GetBlockIdx();
        uint32_t globalBufferIndex = bigCoreDataNum * AscendC::GetBlockIdx();
        this->tileDataNum = tileDataNum;
        if (coreNum < tailBlockNum) {
            this->coreDataNum = bigCoreDataNum;
            this->tileNum = finalBigTileNum;
            this->tailDataNum = bigTailDataNum;
        } else {
            this->coreDataNum = smallCoreDataNum;
            this->tileNum = finalSmallTileNum;
            this->tailDataNum = smallTailDataNum;
            globalBufferIndex -= (bigCoreDataNum - smallCoreDataNum) * (AscendC::GetBlockIdx() - tailBlockNum);
        }
        xGm.SetGlobalBuffer((__gm__ T *)x + globalBufferIndex, this->coreDataNum);
        yGm.SetGlobalBuffer((__gm__ T *)y + globalBufferIndex, this->coreDataNum);
        zGm.SetGlobalBuffer((__gm__ T *)z + globalBufferIndex, this->coreDataNum);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileDataNum * sizeof(T));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileDataNum * sizeof(T));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileDataNum * sizeof(T));
    }


声明Process函数，按Tiling切分次数循环执行CopyIn→Compute→CopyOut计算流程：常规次循环处理Tiling单次最大数据量，最后一次循环适配尾块数据量完成处理。

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc

    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum;
        this->processDataNum = this->tileDataNum;
        for (int32_t i = 0; i < loopCount; i++) {
            if (i == this->tileNum - 1) {
                this->processDataNum = this->tailDataNum;
            }
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }


CopyIn函数依据Tiling实现计算得到的单次处理数据量，完成数据的搬入操作。

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc
private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<T> xLocal = inQueueX.AllocTensor<T>();
        AscendC::LocalTensor<T> yLocal = inQueueY.AllocTensor<T>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileDataNum], this->processDataNum);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileDataNum], this->processDataNum);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }


Compute函数函数依据Tiling实现计算得到的单次处理数据量，完成数据的计算操作。

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<T> xLocal = inQueueX.DeQue<T>();
        AscendC::LocalTensor<T> yLocal = inQueueY.DeQue<T>();
        AscendC::LocalTensor<T> zLocal = outQueueZ.AllocTensor<T>();
        AscendC::Add(zLocal, xLocal, yLocal, this->processDataNum);
        outQueueZ.EnQue<T>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }


CopyOut函数函数依据Tiling实现计算得到的单次处理数据量，完成计算结果的搬出操作。

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<T> zLocal = outQueueZ.DeQue<T>();
        AscendC::DataCopy(zGm[progress * this->tileDataNum], zLocal, this->processDataNum);
        outQueueZ.FreeTensor(zLocal);
    }


定义上文使用的变量

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc
private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::QuePosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::QuePosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<T> xGm;
    AscendC::GlobalTensor<T> yGm;
    AscendC::GlobalTensor<T> zGm;
    uint32_t coreDataNum = 0;
    uint32_t tileNum = 0;
    uint32_t tileDataNum = 0;
    uint32_t tailDataNum = 0;
    uint32_t processDataNum = 0;
};


#### **4.2.2 算子类调用**
算子类实现完成后，通过读取核函数入口的Tiling数据，将其传入算子类对象的Init函数完成算子计算参数的初始化，后续调用Process函数执行算子完整的计算流程。

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc

template <typename T>
__global__ __vector__ void add_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, TilingDataTemplate tiling)
{
    KernelAdd<T> op;
    op.Init(x, y, z, tiling.smallCoreDataNum,
            tiling.bigCoreDataNum, tiling.finalBigTileNum,
            tiling.finalSmallTileNum, tiling.tileDataNum,
            tiling.smallTailDataNum, tiling.bigTailDataNum,
            tiling.tailBlockNum);
    op.Process();
}


### **4.3 Host 侧 `<<<>>>` 调用与验证**
Host侧调用先前完成的 `CalcTiling` 生成 Tiling，然后分配内存后，调用add_custom_template开始计算。
本次测试场景我们使用shape为[33, 31]的half类型数据来进行测试验证。

In [ ]:
%%writefile -a Sources/02.16/add_custom_template.asc

int32_t main(int32_t argc, char *argv[])
{
    uint32_t inputNum = 33 * 31;
    size_t dataSize = inputNum * sizeof(half);

    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    TilingDataTemplate tiling{};
    uint32_t numBlocks = 0;
    CalcTiling<half>(inputNum, tiling, numBlocks);

    uint8_t* xHost;
    uint8_t *xDevice;
    aclrtMallocHost((void**)(&xHost), dataSize);
    aclrtMalloc((void**)&xDevice, dataSize, ACL_MEM_MALLOC_HUGE_FIRST);
    ReadFile("./input/src0.bin", dataSize, xHost, dataSize);
    aclrtMemcpy(xDevice, dataSize, xHost, dataSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t* yHost;
    uint8_t *yDevice;
    aclrtMallocHost((void**)(&yHost), dataSize);
    aclrtMalloc((void**)&yDevice, dataSize, ACL_MEM_MALLOC_HUGE_FIRST);
    ReadFile("./input/src1.bin", dataSize, yHost, dataSize);
    aclrtMemcpy(yDevice, dataSize, yHost, dataSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t *zHost;
    uint8_t *zDevice;
    aclrtMallocHost(reinterpret_cast<void **>(&zHost), dataSize);
    aclrtMalloc(reinterpret_cast<void **>(&zDevice), dataSize, ACL_MEM_MALLOC_HUGE_FIRST);

    add_custom_template<half><<<numBlocks, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(zHost, dataSize, zDevice, dataSize, ACL_MEMCPY_DEVICE_TO_HOST);
    WriteFile("./output/output.bin", zHost, dataSize);

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(xHost);
    aclrtFreeHost(yHost);
    aclrtFreeHost(zHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return 0;
}

执行下面命令来验证算子精度。

In [ ]:
!cd Sources/02.16 && bash run.sh

---
# **课后实践**
请实现泛化的 `sub_custom_template` 算子，算子原型如下：
以shape=[33,31]的float为测试数据，校验算子输出是否符合预期。

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子类型 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" >sub_custom_template</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="3" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">x</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">/</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half, float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND, ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">y</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">/</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half, float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND, ND</td>
  </tr>
  
  <tr>
  <td style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">z</td>
    <td style="padding: 8px;">/</td>
    <td style="padding: 8px;">half, float</td>
    <td style="padding: 8px;">ND, ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

In [ ]:
%%writefile Sources/02.16/sub_custom_template.asc

#include <algorithm>
#include <cstdint>
#include <iostream>
#include <vector>

#include "data_utils.h"
#include "acl/acl.h"
#include "kernel_operator.h"
#include "tiling/platform/platform_ascendc.h"

constexpr int32_t BUFFER_NUM = 1;

struct TilingDataTemplate {
    // TODO: 设计tiling结构体
};

template <typename T>
void CalcTiling(uint32_t inputNum, TilingDataTemplate &tiling, uint32_t &numBlocks)
{
    // TODO: 完成tiling计算
}

template <typename T>
class KernelSub {
public:
    __aicore__ inline KernelSub() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z)
    {
        // TODO:
    }

    __aicore__ inline void Process()
    {
        // TODO:
    }
private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        // TODO:
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        // TODO:
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        // TODO:
    }
private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::QuePosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::QuePosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<T> xGm;
    AscendC::GlobalTensor<T> yGm;
    AscendC::GlobalTensor<T> zGm;
};

template <typename T>
__global__ __vector__ void sub_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, TilingDataTemplate tiling)
{
    // TODO:
}

int32_t main(int32_t argc, char *argv[])
{
    uint32_t inputNum = 33 * 31;
    size_t dataSize = inputNum * sizeof(float);

    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    TilingDataTemplate tiling{};
    uint32_t numBlocks = 0;
    CalcTiling<float>(inputNum, tiling, numBlocks);

    uint8_t* xHost;
    uint8_t *xDevice;
    aclrtMallocHost((void**)(&xHost), dataSize);
    aclrtMalloc((void**)&xDevice, dataSize, ACL_MEM_MALLOC_HUGE_FIRST);
    ReadFile("./input/src0.bin", dataSize, xHost, dataSize);
    aclrtMemcpy(xDevice, dataSize, xHost, dataSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t* yHost;
    uint8_t *yDevice;
    aclrtMallocHost((void**)(&yHost), dataSize);
    aclrtMalloc((void**)&yDevice, dataSize, ACL_MEM_MALLOC_HUGE_FIRST);
    ReadFile("./input/src1.bin", dataSize, yHost, dataSize);
    aclrtMemcpy(yDevice, dataSize, yHost, dataSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t *zHost;
    uint8_t *zDevice;
    aclrtMallocHost(reinterpret_cast<void **>(&zHost), dataSize);
    aclrtMalloc(reinterpret_cast<void **>(&zDevice), dataSize, ACL_MEM_MALLOC_HUGE_FIRST);

    sub_custom_template<float><<<numBlocks, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(zHost, dataSize, zDevice, dataSize, ACL_MEMCPY_DEVICE_TO_HOST);
    WriteFile("./output/output.bin", zHost, dataSize);

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(xHost);
    aclrtFreeHost(yHost);
    aclrtFreeHost(zHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return 0;
}




CMake脚本进行调整，编译sub_custom_template.asc。

In [ ]:
%%writefile Sources/02.16/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture: dav-2201, dav-3510")
find_package(ASC REQUIRED)
project(sub_custom_template LANGUAGES ASC CXX)

add_executable(demo
    sub_custom_template.asc
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

target_link_libraries(demo PRIVATE
    tiling_api
    register
    platform
    m
    dl
)


输入、真值生成脚本进行修改，切换为减法。

In [ ]:
%%writefile Sources/02.16/scripts/gen_data.py
import os
import numpy as np
import argparse

np.random.seed(42)
def gen_golden_data():
    src0 = np.random.randint(-5, 5, [33, 31]).astype(np.float32)
    src1 = np.random.randint(-5, 5, [33, 31]).astype(np.float32)
    golden = src0 - src1
    os.makedirs("input", exist_ok=True)
    os.makedirs("output", exist_ok=True)
    src0.tofile("./input/src0.bin")
    src1.tofile("./input/src1.bin")
    golden.tofile("./output/golden.bin")

if __name__ == "__main__":
    gen_golden_data()

执行下列命令验证精度。

In [ ]:
!cd Sources/02.16 && bash run.sh

**查看答案：**

In [ ]:
!cat ./answer/02.16_answer/sub_custom_template.asc